# DoSA-Maxwell 통합 자동화 튜토리얼

DoSA 설계 파일을 파싱한 후 **`apply_dosa_to_maxwell()`** 함수 한 번으로
Ansys Maxwell에 형상·재질·여자·설정을 모두 적용합니다.

```python
parse_dosa_file()        # DoSA .dsa → DesignModel
Maxwell2d / Maxwell3d()  # pyaedt 세션 생성 (pyaedt 예제와 동일)
apply_dosa_to_maxwell()  # DoSA → Maxwell 변환
```

> **AEDT 버전:** 2026.1 · **모드:** non-graphical · **단위:** mm


In [ ]:
from pathlib import Path
import sys

# -- 경로 설정 --------------------------------------------------------------
ROOT       = Path(r"E:/KDH/gitDosa_Actuator/DoSA-2D/Code/31_DoSA-Maxwell-Automation")
SAMPLE_DIR = Path(r"E:/KDH/gitDosa_Actuator/DoSA-2D/Code/11_DoSA-2D/DoSA-2D/Samples")
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

# -- pyaedt ----------------------------------------------------------------
import ansys.aedt.core

# -- dosa-maxwell: parse + apply 중심 API ----------------------------------
from dosa_maxwell import parse_dosa_file, apply_dosa_to_maxwell, get_profile
from dosa_maxwell.profiles import list_profiles

AEDT_VERSION = "2026.1"   # Maxwell 2026 R1
NG_MODE      = False       # headless (비그래픽)

print(f"ansys.aedt.core : {ansys.aedt.core.__version__}")
print(f"AEDT target     : {AEDT_VERSION}")
print(f"Output dir      : {OUTPUT_DIR}")


ansys.aedt.core : 0.25.0
AEDT target     : 2026.1
Output dir      : E:\KDH\gitDosa_Actuator\DoSA-2D\Code\31_DoSA-Maxwell-Automation\output


c:\Users\user\.ansys_python_venvs\pyMotorEnv_310\lib\site-packages\ansys\aedt\core\visualization\plot\matplotlib.py:49: UserWarning: Graphics dependencies are required. Please install the ``graphics`` target to use this method. You can install it by running `pip install pyaedt[graphics]` or `pip install pyaedt[all]`.
  warnings.warn(ERROR_GRAPHICS_REQUIRED)
c:\Users\user\.ansys_python_venvs\pyMotorEnv_310\lib\site-packages\ansys\aedt\core\visualization\plot\pyvista.py:52: UserWarning: Graphics dependencies are required. Please install the ``graphics`` target to use this method. You can install it by running `pip install pyaedt[graphics]` or `pip install pyaedt[all]`.
  warnings.warn(ERROR_GRAPHICS_REQUIRED)


## 1. DoSA 설계 파싱


In [2]:
design_sol = parse_dosa_file(SAMPLE_DIR / "Solenoid/Solenoid.dsa")
design_vcm = parse_dosa_file(SAMPLE_DIR / "VCM/VCM.dsa")

print(f"Solenoid parts : {[p.name for p in design_sol.parts]}")
print(f"         tests : {[t.name for t in design_sol.tests]}")
print()
print(f"VCM      parts : {[p.name for p in design_vcm.parts]}")
print(f"         tests : {[t.name for t in design_vcm.tests]}")


Solenoid parts : ['coil', 'plunger', 'core', 'case']
         tests : ['force', 'stroke', 'current']

VCM      parts : ['coil', 'magnet', 'plate', 'yoke']
         tests : ['force', 'stroke', 'current']


## 2. Solenoid — Maxwell 2D (MagnetostaticXY)

pyaedt 예제와 동일하게 세션을 열고,
`apply_dosa_to_maxwell()` 한 번으로 DoSA 모델을 적용합니다.


In [3]:
profile = get_profile("le01_2020r1")   # Magnetostatic 프리셋

m2d_sol = ansys.aedt.core.Maxwell2d(
    design="Solenoid_2D",
    solution_type="MagnetostaticXY",
    version=AEDT_VERSION,
    non_graphical=NG_MODE,
    new_desktop=True,
)


PyAEDT INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.25.0.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: AEDT version 2026.1.
PyAEDT INFO: New AEDT session is starting on gRPC port 53492.
PyAEDT INFO: Starting new AEDT gRPC session on port 53492.
PyAEDT INFO: Launching AEDT server with gRPC transport mode: wnua
PyAEDT INFO: Electronics Desktop started on gRPC port 53492 after 24.8 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v261\AnsysEM
PyAEDT INFO: Connected to AEDT gRPC session on port 53492.
PyAEDT INFO: Project Project29 has been created.
PyAEDT INFO: Added design 'Solenoid_2D' of type Maxwell 2D.
PyAEDT INFO: AEDT objects correctly read


In [4]:
m2d_sol.modeler.model_units = "mm"

# DoSA -> Maxwell: 형상/재질/코일 여자/해석 설정 모두 적용
result = apply_dosa_to_maxwell(design_sol, m2d_sol, profile=profile)
print(result)
for e in result.get("errors", []):
    print(" !", e)
# 


PyAEDT INFO: Modeler2D class has been initialized!
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 1sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Boundary Coil Coil_coil has been created.
<ApplyResult OK | parts=4 errors=0>


In [ ]:
out = OUTPUT_DIR / "solenoid_2d" / "Solenoid.aedt"
out.parent.mkdir(parents=True, exist_ok=True)

m2d_sol.save_project(str(out))
print("Saved:", m2d_sol.project_file)
# m2d_sol.release_desktop(close_projects=False, close_desktop=False)


## 3. VCM — Maxwell 2D (MagnetostaticXY)


In [5]:
m2d_vcm = ansys.aedt.core.Maxwell2d(
    design="VCM_2D",
    solution_type="MagnetostaticXY",
    version=AEDT_VERSION,
    non_graphical=NG_MODE,
    new_desktop=False,
)
m2d_vcm.modeler.model_units = "mm"

result_vcm = apply_dosa_to_maxwell(design_vcm, m2d_vcm, profile=profile)
print(result_vcm)
for e in result_vcm.get("errors", []):
    print(" !", e)

out_vcm = OUTPUT_DIR / "vcm_2d" / "VCM.aedt"
out_vcm.parent.mkdir(parents=True, exist_ok=True)
m2d_vcm.save_project(str(out_vcm))
print("Saved:", m2d_vcm.project_file)
# m2d_vcm.release_desktop(close_projects=False, close_desktop=False)


PyAEDT INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.25.0.
PyAEDT INFO: Returning found Desktop session with PID 30052!
PyAEDT INFO: No project is defined. Project Project29 exists and has been read.
PyAEDT INFO: Added design 'VCM_2D' of type Maxwell 2D.
PyAEDT INFO: AEDT objects correctly read
PyAEDT INFO: Modeler2D class has been initialized!
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Boundary Coil Coil_coil has been created.
<ApplyResult OK | parts=4 errors=0>
PyAEDT INFO: Project VCM Saved correctly
Saved: E:\KDH\gitDosa_Actuator\DoSA-2D\Code\31_DoSA-Maxwell-Automation\output\vcm_2d\VCM.aedt


## 4. Solenoid — Maxwell 3D (축대칭 → 360° 회전체)

DoSA 2D 단면(X=반경, Y=축)을 Y축 기준 360° 회전시켜 3D 솔리드를 생성합니다.
`apply_dosa_to_maxwell()`이 Maxwell3d 인스턴스를 받으면 자동으로 3D 처리합니다.


In [6]:
m3d_sol = ansys.aedt.core.Maxwell3d(
    design="Solenoid_3D",
    solution_type="Magnetostatic",
    version=AEDT_VERSION,
    non_graphical=NG_MODE,
    new_desktop=False,
)
m3d_sol.modeler.model_units = "mm"

# Maxwell3d 감지 -> polyline 후 sweep_around_axis(Y, 360deg) 자동 적용
result_3d = apply_dosa_to_maxwell(design_sol, m3d_sol, profile=profile)
print(result_3d)
for e in result_3d.get("errors", []):
    print(" !", e)

out_sol3d = OUTPUT_DIR / "solenoid_3d" / "Solenoid_3D.aedt"
out_sol3d.parent.mkdir(parents=True, exist_ok=True)
m3d_sol.save_project(str(out_sol3d))
print("Saved:", m3d_sol.project_file)
# m3d_sol.release_desktop(close_projects=False, close_desktop=False)


PyAEDT INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.25.0.
PyAEDT INFO: Returning found Desktop session with PID 30052!
PyAEDT INFO: No project is defined. Project VCM exists and has been read.
PyAEDT INFO: Added design 'Solenoid_3D' of type Maxwell 3D.
PyAEDT INFO: AEDT objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
<ApplyResult OK | parts=4 errors=0>
PyAEDT INFO: Project Solenoid_3D Saved correctly
Saved: E:\KDH\gitDosa_Actuator\DoSA-2D\Code\31_DoSA-Maxwell-Automation\output\solenoid_3d\Solenoid_3D.aedt


## 5. VCM — Maxwell 3D


In [7]:
m3d_vcm = ansys.aedt.core.Maxwell3d(
    design="VCM_3D",
    solution_type="Magnetostatic",
    version=AEDT_VERSION,
    non_graphical=NG_MODE,
    new_desktop=False,
)
m3d_vcm.modeler.model_units = "mm"

result_vcm3d = apply_dosa_to_maxwell(design_vcm, m3d_vcm, profile=profile)
print(result_vcm3d)
for e in result_vcm3d.get("errors", []):
    print(" !", e)

out_vcm3d = OUTPUT_DIR / "vcm_3d" / "VCM_3D.aedt"
out_vcm3d.parent.mkdir(parents=True, exist_ok=True)
m3d_vcm.save_project(str(out_vcm3d))
print("Saved:", m3d_vcm.project_file)
# m3d_vcm.release_desktop(close_projects=False, close_desktop=False)


PyAEDT INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.25.0.
PyAEDT INFO: Returning found Desktop session with PID 30052!
PyAEDT INFO: No project is defined. Project Solenoid_3D exists and has been read.
PyAEDT INFO: Added design 'VCM_3D' of type Maxwell 3D.
PyAEDT INFO: AEDT objects correctly read
PyAEDT INFO: Modeler class has been initialized! Elapsed time: 0m 0sec
PyAEDT INFO: Materials class has been initialized! Elapsed time: 0m 0sec
<ApplyResult OK | parts=4 errors=0>
PyAEDT INFO: Project VCM_3D Saved correctly
Saved: E:\KDH\gitDosa_Actuator\DoSA-2D\Code\31_DoSA-Maxwell-Automation\output\vcm_3d\VCM_3D.aedt


## 6. PDF 기반 프로파일 시뮬레이션 (2020R1 + 2014)

추가된 PDF들에 대응되는 프로파일을 `unified_plan.json`에서 자동 탐색해
동일한 DoSA 모델에 적용하고 프로젝트를 생성합니다.

- 2020R1: `le01_2020r1`, `ws01_2020r1`
- 2014: `coupling_1114_2014`, `tpc_1210_2014`

필요하면 `RUN_SOLVE = True`로 바꿔 실제 해석까지 실행할 수 있습니다.


In [ ]:
PDF_ROOT = ROOT.parents[1]
RUN_SOLVE = False  # True로 변경하면 setup 해석까지 수행

all_profiles = [p for p in list_profiles() if p.get("source_pdf") and p["source_pdf"] != "internal"]
selected_profiles = []
for p in all_profiles:
    source_pdf = p["source_pdf"]
    pdf_path = PDF_ROOT / source_pdf
    if pdf_path.exists() and ("2020" in source_pdf or "2014" in source_pdf):
        selected_profiles.append((p["name"], pdf_path))

print("Detected PDF profiles:")
for name, path in selected_profiles:
    print(f" - {name}: {path.name}")

for i, (profile_name, pdf_path) in enumerate(selected_profiles):
    prof = get_profile(profile_name)

    print(f"\n=== {profile_name} | solution={prof.solution_type} ===")
    print(f"PDF exists: {pdf_path.exists()} -> {pdf_path}")

    m2d_pdf = ansys.aedt.core.Maxwell2d(
        design=f"Solenoid_2D_{profile_name}",
        solution_type=("TransientXY" if prof.solution_type == "Transient" else "MagnetostaticXY"),
        version=AEDT_VERSION,
        non_graphical=NG_MODE,
        new_desktop=(i == 0),
    )
    m2d_pdf.modeler.model_units = "mm"

    result_pdf = apply_dosa_to_maxwell(design_sol, m2d_pdf, profile=prof)
    print(result_pdf)
    for e in result_pdf.get("errors", []):
        print(" !", e)

    out_pdf = OUTPUT_DIR / "pdf_profiles" / f"Solenoid_{profile_name}.aedt"
    out_pdf.parent.mkdir(parents=True, exist_ok=True)
    m2d_pdf.save_project(str(out_pdf))
    print("Saved:", m2d_pdf.project_file)

    if RUN_SOLVE:
        try:
            m2d_pdf.analyze_setup("DoSA_Setup")
            print("Solve done: DoSA_Setup")
        except Exception as exc:
            print(f"Solve skipped/failed: {exc}")


## 7. 세션 정리 및 결과 확인


In [ ]:
# 공유 Desktop 세션 정리
# 참고: Desktop()를 직접 호출하면 활성 세션이 없을 때 새 세션이 열릴 수 있어,
# 이미 생성한 앱 객체에서 release_desktop()를 호출합니다.
apps = [
    globals().get("m2d_pdf"),
    globals().get("m3d_vcm"),
    globals().get("m3d_sol"),
    globals().get("m2d_vcm"),
    globals().get("m2d_sol"),
]

released = False
for app in apps:
    if app is None:
        continue
    try:
        app.release_desktop(close_projects=True, close_desktop=True)
    except TypeError:
        app.release_desktop()
    released = True
    print("AEDT desktop released.")
    break

if not released:
    print("No active AEDT app instance found; cleanup skipped.")

print("\n=== 생성된 .aedt 파일 ===")
for f in sorted(OUTPUT_DIR.rglob("*.aedt")):
    print(f"  {f}")


PyAEDT INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.25.0.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: AEDT version 2025.2.
PyAEDT INFO: New AEDT session is starting on gRPC port 50829.
PyAEDT INFO: Starting new AEDT gRPC session on port 50829.
PyAEDT INFO: Launching AEDT server with gRPC transport mode: wnua
PyAEDT INFO: Electronics Desktop started on gRPC port 50829 after 50.5 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v252\AnsysEM
PyAEDT INFO: Connected to AEDT gRPC session on port 50829.
PyAEDT WARNING: Service Pack is not detected. PyAEDT is currently connecting in Insecure Mode.
PyAEDT WARNING: Please download and install latest Service Pack to use connect to AEDT in Secure Mode.
PyAEDT INFO: Desktop has been released and closed.
Cleanup skipped: Desktop.release_desktop() got an unexpected keyword argument 'close_desktop'

=== 생성된 .aedt 파일 ===
  E: